# Notebook 03 : SQL Analysis via SQLite

**Goal:** Run all SQL queries from the `/sql` folder using Python + SQLite.


In [ ]:
import pandas as pd
import sqlite3

df = pd.read_csv('../data/superstore_clean.csv', parse_dates=['Order Date', 'Ship Date'])

conn = sqlite3.connect(':memory:')
df.columns = [c.replace(' ', '_').replace('-', '_') for c in df.columns]
df.to_sql('orders', conn, if_exists='replace', index=False)
print(f'Loaded {len(df)} rows into SQLite orders table.')
print(f'Columns: {list(df.columns)}')

## Query 1 : Top profitable sub-categories

In [ ]:
q1 = """
SELECT Sub_Category,
       ROUND(SUM(Profit), 2)          AS total_profit,
       ROUND(AVG(Profit_Margin)*100,1) AS avg_margin_pct,
       COUNT(*)                        AS order_count
FROM orders
GROUP BY Sub_Category
ORDER BY total_profit DESC
LIMIT 10;
"""
pd.read_sql(q1, conn)

## Query 2 : Regional performance

In [ ]:
q2 = """
SELECT Region,
       ROUND(SUM(Sales), 2)    AS total_sales,
       ROUND(SUM(Profit), 2)   AS total_profit,
       ROUND(AVG(Profit_Margin)*100, 1) AS avg_margin_pct,
       COUNT(DISTINCT Customer_ID) AS unique_customers
FROM orders
GROUP BY Region
ORDER BY total_profit DESC;
"""
pd.read_sql(q2, conn)

## Query 3 : Monthly sales trend

In [ ]:
q3 = """
SELECT strftime('%Y-%m', Order_Date) AS month,
       ROUND(SUM(Sales), 2)           AS monthly_sales,
       ROUND(SUM(Profit), 2)          AS monthly_profit,
       COUNT(DISTINCT Order_ID)       AS orders
FROM orders
GROUP BY month
ORDER BY month;
"""
pd.read_sql(q3, conn).tail(12)

## Query 4 : Discount impact on margin

In [ ]:
q4 = """
SELECT
  CASE
    WHEN Discount = 0        THEN '0% (no discount)'
    WHEN Discount <= 0.1     THEN '1-10%'
    WHEN Discount <= 0.2     THEN '11-20%'
    WHEN Discount <= 0.3     THEN '21-30%'
    ELSE '> 30%'
  END AS discount_bucket,
  COUNT(*) AS orders,
  ROUND(AVG(Profit_Margin)*100, 1) AS avg_margin_pct,
  ROUND(SUM(Profit), 2) AS total_profit
FROM orders
GROUP BY discount_bucket
ORDER BY avg_margin_pct DESC;
"""
pd.read_sql(q4, conn)

## Query 5 : Loss-making products

In [ ]:
q5 = """
SELECT Product_Name,
       Sub_Category,
       ROUND(SUM(Profit), 2)  AS total_profit,
       COUNT(*) AS times_ordered
FROM orders
GROUP BY Product_Name, Sub_Category
HAVING total_profit < -500
ORDER BY total_profit ASC
LIMIT 10;
"""
pd.read_sql(q5, conn)

## Query 6 : Window function: running total by region

This query demonstrates advanced SQL. Include this in your portfolio to stand out.

In [ ]:
q6 = """
SELECT Region,
       strftime('%Y-%m', Order_Date) AS month,
       ROUND(SUM(Sales), 2) AS monthly_sales,
       ROUND(SUM(SUM(Sales)) OVER (
           PARTITION BY Region
           ORDER BY strftime('%Y-%m', Order_Date)
       ), 2) AS running_total
FROM orders
GROUP BY Region, month
ORDER BY Region, month
LIMIT 20;
"""
pd.read_sql(q6, conn)

## Query 7 : Customer segment analysis

In [ ]:
q7 = """
SELECT Segment,
       COUNT(DISTINCT Customer_ID) AS customers,
       COUNT(DISTINCT Order_ID)    AS orders,
       ROUND(SUM(Sales), 2)        AS total_sales,
       ROUND(SUM(Profit), 2)       AS total_profit,
       ROUND(AVG(Profit_Margin)*100, 1) AS avg_margin_pct,
       ROUND(SUM(Sales) / COUNT(DISTINCT Customer_ID), 2) AS sales_per_customer
FROM orders
GROUP BY Segment
ORDER BY total_profit DESC;
"""
pd.read_sql(q7, conn)